[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q4/02_cross_host_evaluation.ipynb)

# R2-Q4 Week 3 — Measure cross-host transfer

### This notebook produces `gap_table.parquet`, the within-host vs cross-host gap that the interpretation step reads next week.

This notebook takes the classifier you trained in Week 2 and applies it to each multi-host disease on its held-out host — the cross-host evaluation. Comparing the resulting cross-host accuracy to the within-host accuracy from Week 2 gives you the gap. A small gap suggests the classifier learned generalizable disease features; a large gap suggests it leaned on host-specific cues. Per-disease accuracy and gap numbers feed directly into next week's interpretation.

By the end of this notebook you will have:

- Per-disease cross-host accuracy saved as `cross_host_accuracy.parquet`.
- A per-disease gap table (within − cross) saved as `gap_table.parquet`, aggregated per your Precommit 3 decision.
- A `comparison_summary.json` with the headline gap figure, the per-disease breakdown, and the input list for the draft presentation.
- Submitted draft presentation.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of `precommit.json` and the trained classifier

In Week 2 you trained a classifier on most of the hosts for each disease and measured how well it does *within* host — on held-out images of those same hosts. This week you give it the test it was built for: the **withheld host** for each disease, the one host NB01 set aside and never trained on. Running the classifier there and comparing to the within-host number gives you the **gap** — the heart of R2-Q4.

A small gap means the classifier recognizes the disease about as well on a new host as on familiar ones: it learned the disease. A large gap means accuracy fell off when the host changed: it leaned on something host-specific. The gap is measured disease by disease, because the answer can differ from one disease to the next, and that per-disease pattern is what next week's interpretation is built on.

This notebook reads back two things before it does anything else: the **precommit** (the design decisions from Week 1 — which diseases, and which host is withheld for each) and the **Week-2 records** NB01 saved (the trained classifier, the within-host accuracy for each disease, and the exact label order the classifier was trained on). Reading the label order back rather than rebuilding it from scratch is what guarantees the model's output indices mean the same diseases here as they did in training — get that wrong and every number this week is quietly meaningless.

In [ ]:
# Read back the three decisions you locked in NB00. The load is defensive: a
# missing or half-written file should fail here, with a clear message, rather
# than partway through the evaluation.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
if not precommit_path.exists():
    raise FileNotFoundError(
        f"No precommit.json at {precommit_path}. Run NB00 "
        "(00_orient_and_precommit.ipynb) to completion first — it writes the "
        "Week-1 decisions this notebook reads."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# All three Week-1 decisions should be present before Week 3 starts.
for key in ("disease_list", "hold_out_scheme", "cross_host_aggregation"):
    if precommit.get(key) is None:
        raise KeyError(
            f"precommit.json is missing '{key}'. Re-run NB00 — every decision "
            "should be filled in before you reach Week 3."
        )

# Compose the label space exactly as NB01 did when it trained the classifier:
# the multi-host diseases, then any single-host control diseases, then "healthy"
# if it was included. The classifier predicts among ALL of these, and the ORDER
# here defines its output-index space — so it has to match NB01's order. The
# next cell asserts it does, against the label order NB01 saved.
dl = precommit["disease_list"]
diseases = list(dl["multi_host_diseases"]) + list(dl["control_single_host_diseases"])
if dl.get("include_healthy"):
    diseases = diseases + ["healthy"]

scheme    = precommit["hold_out_scheme"]["scheme"]
held_out  = precommit["hold_out_scheme"]["held_out_host"]
agg_level = precommit["cross_host_aggregation"]["level"]

# The transfer subjects are the diseases that have a withheld host — the only
# ones with a cross-host test to run and a gap to report. A single-host control
# disease is in the label space (the classifier learns it) but has no withheld
# host, so it gets no cross-host slice. With the default precommit (no controls)
# these two sets are identical; keeping them separate keeps the notebook correct
# if you chose to keep a control disease.
transfer_diseases = [d for d in diseases if d in held_out]

disease_to_idx = {d: i for i, d in enumerate(diseases)}
idx_to_disease = {i: d for d, i in disease_to_idx.items()}
num_classes = len(diseases)

# A single fixed-host classifier is what NB01 trained and what this notebook
# evaluates. Leave-one-host-out would mean a classifier per held-out
# configuration and a different evaluation — flag it rather than quietly running
# the wrong shape.
if scheme != "fixed_host":
    print(
        f"WARNING: hold_out_scheme is {scheme!r}, not 'fixed_host'. This "
        "notebook evaluates one classifier on one withheld host per disease; "
        "leave-one-host-out produces a classifier per configuration and a "
        "different evaluation. Check with your mentor before continuing.\n"
    )

print("Loaded precommit.json")
print(f"  Label space ({num_classes}): {', '.join(diseases)}")
print(f"  Hold-out scheme: {scheme}")
print(f"  Transfer subjects ({len(transfer_diseases)}) — each gets a cross-host test:")
for d in transfer_diseases:
    print(f"    {d:18s} -> withheld host: {held_out[d]}")
controls = [d for d in diseases if d not in held_out]
if controls:
    print(f"  Controls (in the label space, no cross-host test): {', '.join(controls)}")
print(f"  Aggregation level: {agg_level}")

Now load what NB01 produced. Three files:

- `classifier.pt` — the trained weights. Section 3 rebuilds the model from them and runs it on the withheld hosts. (Loading it here, early, just confirms the file is present and readable; the model itself is rebuilt later.)
- `within_host_summary.json` — NB01's record of the run. You read three things from it: the **label order** (the diseases in the exact order the classifier's output indices follow), `num_classes`, and the `DEV_MODE` setting NB01 trained under. The label order is the load-bearing one — the cell below asserts the label space this notebook just composed is identical to the one NB01 saved.
- `within_host_accuracy.parquet` — the per-disease within-host accuracy, carrying the near-chance flags NB01's gate set. This is the within-host side of every gap you compute in Section 4; a flagged disease's gap will carry that caveat forward.

The load stays defensive: a missing or half-written file should stop the notebook here, naming the file, rather than failing deep inside the evaluation.

In [ ]:
import json
import torch
import pandas as pd

# 1. The trained classifier — a state_dict NB01 saved with torch.save. Load it
#    to CPU now just to confirm it's present and readable; Section 3 moves the
#    rebuilt model to the GPU.
classifier_path = OUTPUT_DIR / "classifier.pt"
if not classifier_path.exists():
    raise FileNotFoundError(
        f"No classifier.pt at {classifier_path}. Run NB01 "
        "(01_within_host_classifier.ipynb) to completion first — it trains and "
        "saves the classifier this notebook evaluates."
    )
state_dict = torch.load(classifier_path, map_location="cpu")

# 2. NB01's within-host summary — carries the label order, num_classes, the
#    DEV_MODE variant NB01 trained under, and your Week-2 gate interpretation.
summary_path = OUTPUT_DIR / "within_host_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        f"No within_host_summary.json at {summary_path}. Re-run NB01 — it "
        "writes the record this notebook reads the label order back from."
    )
with open(summary_path) as f:
    within_host_summary = json.load(f)

# 3. The per-disease within-host accuracy table — the within-host side of the
#    gap (Section 4), with the gate flags NB01 set.
wh_acc_path = OUTPUT_DIR / "within_host_accuracy.parquet"
if not wh_acc_path.exists():
    raise FileNotFoundError(
        f"No within_host_accuracy.parquet at {wh_acc_path}. Re-run NB01 — it "
        "writes the within-host accuracy this notebook compares against."
    )
within_host_df = pd.read_parquet(wh_acc_path)

# --- The load-bearing consistency check. The label space this notebook composed
#     from the precommit must be the exact one NB01 trained on, in the same
#     order. If it isn't, the model's output index 0 (say) means one disease here
#     and a different one in training, and every accuracy and gap is wrong --
#     silently. Fail loud instead. ---
saved_diseases = within_host_summary["methodology"]["diseases"]
assert diseases == saved_diseases, (
    "Label-space mismatch between this notebook and NB01.\n"
    f"  NB02 composed from precommit: {diseases}\n"
    f"  NB01 saved after training:    {saved_diseases}\n"
    "These must be identical — same diseases, same order. If you changed the "
    "disease list since training, re-run NB00 and NB01 before this notebook."
)
assert within_host_summary["methodology"]["num_classes"] == num_classes, (
    f"num_classes disagrees with NB01 "
    f"({within_host_summary['methodology']['num_classes']} saved vs {num_classes} "
    "here). Re-run NB01."
)

# Within-host accuracy and gate flag, looked up by disease, for Section 4's gap.
within_host_acc  = within_host_df.set_index("disease")["within_host_acc"].to_dict()
within_host_flag = within_host_df.set_index("disease")["flag"].to_dict()

print(f"Loaded classifier.pt ({len(state_dict)} weight tensors)")
print(f"Loaded within-host records — label order confirmed against NB01 "
      f"({len(saved_diseases)} diseases)")
print(f"  Week-2 overall within-host accuracy: "
      f"{within_host_summary['results']['overall_within_host_acc']:.3f}")
flagged = within_host_summary["results"].get("flagged_diseases", [])
if flagged:
    print(f"  Near-chance flags carried from NB01: {', '.join(flagged)}")
else:
    print("  No near-chance flags from NB01.")

One setting to fix before evaluating. `DEV_MODE` does just one thing in this notebook: it picks which size of PlantVillage the cross-host images come from. There's no training here — the classifier is already trained — so there's no epoch count to set.

The one rule: `DEV_MODE` here should match the setting NB01 trained under. The gap compares a within-host number (from NB01) against a cross-host number (computed here); if the two come from different sizes of the dataset, that comparison is apples to oranges. NB01 recorded which setting it used, and the cell below warns you if this run disagrees.

Running the model forward still needs a GPU, so the cell checks for one and stops early, with instructions, if it's missing.

In [ ]:
# Run configuration. NB02 only runs the trained model forward (no training), so
# DEV_MODE controls one thing here: which size of PlantVillage the cross-host
# images are loaded from. It should match the size NB01 trained under, or you'd
# be comparing a within-host number from one dataset against a cross-host number
# from another.
DEV_MODE = True
PV_VARIANT = "tiny" if DEV_MODE else "full"

# Seed anything outside the model (data loading, evaluation ordering) for
# reproducibility. The evaluation loader runs shuffle=False, so the score is
# deterministic regardless, but seeding keeps the whole notebook consistent.
iri.seed_all(42)

# evaluate_in_categories moves the model to the GPU for its forward pass, so a
# GPU is required here too. Fail now, with a clear message.
assert iri.has_gpu(), (
    "No GPU detected. Running the classifier needs a GPU. In Colab: "
    "Runtime -> Change runtime type -> T4 GPU, then re-run from the top."
)

# The within-host numbers you'll compare against were produced on whatever
# variant NB01 used. Warn if this run disagrees — the gap is only meaningful
# when both sides come from the same data.
trained_dev_mode = within_host_summary["methodology"].get("dev_mode")
if trained_dev_mode is not None and trained_dev_mode != DEV_MODE:
    print(
        f"WARNING: NB01 trained with DEV_MODE = {trained_dev_mode}, but this run "
        f"has DEV_MODE = {DEV_MODE}. The within-host accuracy you'll compare "
        "against came from the other setting. Match them before reading the gap "
        "as a result.\n"
    )

print(f"DEV_MODE   = {DEV_MODE}")
print(f"PV_VARIANT = {PV_VARIANT!r}")
print("Seed set to 42; GPU detected.")

### 2) Build the cross-host evaluation set (the same diseases, on the held-out hosts)

In Week 2, for each disease, you split its hosts into a training group and one withheld host. NB01 trained on the training hosts. The withheld host was set aside, never seen during training — that is the host you score now.

Building this slice is the mirror image of NB01's hold-out. There, you kept the rows whose host was *not* the withheld host. Here, you keep exactly the rows you dropped: for each disease, the images whose host *is* its withheld host. Same diseases, opposite side of the split.

Two things carry over from NB01 unchanged, because the model expects them:

- **Relabel to the disease index.** The classifier predicts a disease index, 0 to N−1, in the order of your label space. The cross-host rows have to carry that same index in their `class_idx` column, the same remap NB01 applied to its training frame — otherwise the image loader hands the model PlantVillage's original class numbers and the scoring is nonsense.
- **Don't reset the index.** The metadata frame's index is how each row is matched to its image. Leave it alone all the way through.

Only the diseases with a withheld host — your transfer subjects — get a slice. A single-host control disease, if you kept one, has no other host to transfer to, so it isn't part of this evaluation.

In [ ]:
# Load PlantVillage at the size DEV_MODE selected — the same variant NB01 used
# (Section 1 warned you if they disagree). The loader returns a metadata table
# (one row per image) and the images. Do NOT reset_index on the metadata
# anywhere below — the frame's index is how PlantVillageDataset lines each row
# up with its image.
meta, hf = iri.load_plantvillage(variant=PV_VARIANT)

print(f"PlantVillage ({PV_VARIANT}): {len(meta):,} images")
print("A few rows (host / disease / split):")
print(meta[["host", "disease", "split"]].head())

### Practice 2.1 — Keep the withheld host for each disease

You need a boolean mask, one value per row of `in_scope`, that is `True` exactly when that row's host is the host withheld for that row's disease. This is the inverse of NB01's `is_training_host`: there the mask was `True` when the host was *not* the withheld one; here it's `True` when the host *is* the withheld one.

**Hint.** The withheld host depends on the disease — `held_out["Late_blight"]` may differ from `held_out["Early_blight"]` — so this is a per-row comparison, not one fixed host. For each row you're asking: *does this row's host equal the host withheld for this row's disease?* A row-wise way to get there is to map each row's disease to its withheld host (`in_scope["disease"].map(held_out)`) and compare that, element-wise, to `in_scope["host"]`.

If you leave the fill blank, the line that builds `cross_host_frame` raises `NameError`.

In [ ]:
import pandas as pd

# Rows for your transfer subjects only — the diseases that have a withheld host.
# (Controls, if any, are excluded here: no withheld host, no cross-host test.)
in_scope = meta[meta["disease"].isin(transfer_diseases)]

# Practice 2.1 — define `is_withheld_host`: True where this row's host IS the
# host withheld for that row's disease. See the hint in the markdown above.
# (If you leave this blank, the next line raises NameError.)
# is_withheld_host = ...

cross_host_frame = in_scope[is_withheld_host].copy()

# Relabel to the disease index — the same map NB01 used on its training frame.
# Do NOT reset_index. PlantVillageDataset reads class_idx straight off this
# frame, so this line is what puts predictions and truth in the same disease
# space.
cross_host_frame["class_idx"] = cross_host_frame["disease"].map(disease_to_idx)

print(f"Cross-host evaluation set: {len(cross_host_frame):,} images "
      f"across {len(transfer_diseases)} transfer subject(s)")

In [ ]:
# Inspect before trusting, the same three checks NB01 ran on its training pool,
# pointed the other way: the slice should be exactly the withheld hosts, every
# transfer subject should be present, and every one should have images to score.

# 1. Every row really is a withheld host — nothing from the training hosts leaked in.
for d in transfer_diseases:
    rows = cross_host_frame[cross_host_frame["disease"] == d]
    hosts = set(rows["host"])
    assert hosts == {held_out[d]}, (
        f"{d}: cross-host slice contains hosts {hosts}, expected only "
        f"{{{held_out[d]}}}. Check your Practice 2.1 mask."
    )

# 2. Every transfer subject is present in the slice.
present = set(cross_host_frame["disease"])
missing = [d for d in transfer_diseases if d not in present]
assert not missing, (
    f"No cross-host images for {missing}. The withheld host for these diseases "
    f"has no images at the {PV_VARIANT!r} variant — try DEV_MODE = False, or "
    "revisit the held_out_host choice in NB00."
)

# 3. The relabelled class_idx lands inside the model's output space (0..N-1).
#    It's a subset, not all of 0..N-1, because controls (if any) aren't here.
assert set(cross_host_frame["class_idx"]).issubset(set(range(num_classes))), (
    "Relabelled class_idx falls outside 0..num_classes-1. Check disease_to_idx "
    "and the remap line."
)

# A per-disease breakdown: the withheld host scored, and how many images.
print(f"{'disease':18s} {'withheld host':14s} {'cross-host images':>18s}")
for d in transfer_diseases:
    n = int((cross_host_frame["disease"] == d).sum())
    print(f"{d:18s} {held_out[d]:14s} {n:>18,}")
print(f"{'TOTAL':18s} {'':14s} {len(cross_host_frame):>18,}")

### 3) Evaluate cross-host: per-disease accuracy on the cross-host samples

The classifier is trained and the cross-host slice is built, so scoring it is the routine you already wrote in Week 2. `predict` takes the trained weights and an evaluation frame, runs the model over every image with the deterministic eval transform, and returns two arrays: the true disease index of each image and the disease the model predicted. It's reproduced below unchanged — the only difference from Week 2 is the frame you hand it: the withheld-host slice instead of the within-host test split.

Both arrays are disease indices in the same 0-to-N−1 space, because the model was trained on the disease label space and the slice was relabelled to match. That's what lets you compare predictions to truth directly, with no translation step.

In [ ]:
# The same routine you wrote in Week 2, unchanged. It rebuilds the model from
# the saved weights, runs it over the evaluation frame under the deterministic
# eval transform (shuffle=False, so the result is reproducible), and returns the
# true and predicted disease index for every image. This week you call it on the
# cross-host slice; nothing else changes. Both arrays are disease indices,
# 0..N-1, so predictions and truth are directly comparable — no remapping.
import torch
from torch.utils.data import DataLoader

def predict(state_dict, eval_metadata, hf_dataset, *, num_classes, batch_size=32):
    model = iri.build_baseline_model(num_classes, pretrained=False)
    model.load_state_dict(state_dict)
    model = model.cuda().eval()

    ds = iri.PlantVillageDataset(
        eval_metadata, hf_dataset, transform=iri.imagenet_eval_transform()
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)

    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.cuda()).argmax(dim=1).cpu()
            y_pred.append(preds)
            y_true.append(labels)
    return torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()

y_true, y_pred = predict(state_dict, cross_host_frame, hf, num_classes=num_classes)
print(f"Scored {len(y_true):,} cross-host images.")

### Practice 3.1 — Per-disease cross-host accuracy

This is the number R2-Q4 turns on. For each disease, compute the fraction of its cross-host images the model classified correctly — exactly the per-disease accuracy you wrote in Week 2 (Practice 4.1), now on the cross-host arrays instead of the within-host ones.

The definition, the same on both sides of the gap: of the images whose **true** label is disease `d`, what fraction did the model **predict** as `d`? The loop below isolates the rows whose true label is `d` (`y_true == i`); your line computes the fraction of those the model got right.

Per-disease, not overall, for the reason the page gives: an overall cross-host number averages the diseases that transfer well with the ones that don't, hiding exactly the pattern next week's interpretation reads. The overall number is computed too, but only as context.

Replace the `NotImplementedError` with the accuracy line. If you leave it, the cell stops there.

In [ ]:
import numpy as np

# Overall cross-host accuracy — context, not the headline. The per-disease
# numbers below are the result.
overall_cross_host_acc = float((y_pred == y_true).mean())

cross_host_accuracy = {}
cross_host_n = {}
for i, d in idx_to_disease.items():
    if d not in transfer_diseases:
        continue                          # controls have no cross-host slice
    is_this_disease = (y_true == i)       # rows whose TRUE label is disease d
    cross_host_n[d] = int(is_this_disease.sum())

    # Practice 3.1 — set cross_host_accuracy[d] to the fraction of disease d's
    # cross-host rows the model predicted as d. Same computation as NB01
    # Practice 4.1, on these arrays. See the hint in the markdown above.
    raise NotImplementedError(
        "Practice 3.1 — set cross_host_accuracy[d] to disease d's cross-host "
        "accuracy: of the rows whose true label is d, the fraction predicted as d."
    )
    # cross_host_accuracy[d] = ...

print("Per-disease cross-host accuracy computed.")

Read the cross-host numbers back as a table before the comparison. Two things to eyeball, the same habits from Week 2: each disease has enough cross-host images behind its number that the accuracy isn't just noise, and the values aren't all pinned at 0.0 or 1.0 (which usually points to something upstream, not a perfect or useless model). These are the cross-host side of the gap you build in Section 4.

A note on the function's vocabulary you'll meet next: the per-disease accuracy here is what some of the shared library code calls a "category" accuracy. In R2-Q4 the category *is* the disease — there's no coarser grouping — so read "per-category" as "per-disease" wherever it appears.

In [ ]:
print(f"{'disease':18s} {'cross-host acc':>16s} {'n images':>10s}")
for d in transfer_diseases:
    print(f"{d:18s} {cross_host_accuracy[d]:>16.3f} {cross_host_n[d]:>10,}")
print(f"{'OVERALL':18s} {overall_cross_host_acc:>16.3f} {len(y_true):>10,}")

if DEV_MODE:
    print("\nDEV_MODE is on — these numbers come from a small dev run. Re-run "
          "with DEV_MODE = False (matching NB01) before reading them as results.")

### 4) Compute the gap: per-disease (within − cross), aggregated per Precommit 3

Now the comparison the notebook was built for. For each disease you have two numbers: how well the classifier did on familiar hosts (within-host, from Week 2) and how well it did on the withheld host (cross-host, from Section 3). The **gap** is the first minus the second.

A gap near zero means the classifier did about as well on the new host as on familiar ones — it recognized the disease, not the host. A large positive gap means accuracy dropped on the new host — the classifier was leaning on something specific to the training hosts that didn't carry over. A gap can even come out slightly negative (better on the withheld host than within host); that's usually noise on a small test set, worth noting, not over-reading.

You build the gap **per disease**, because that's where the signal is. One overall gap would average a disease that transfers cleanly against one that collapses, and the page's whole prediction is that diseases differ here. Whether you also report an overall number is your Precommit 3 choice — the per-disease rows are always built; an overall row is added only if you precommitted to it.

One caution carries over from Week 2. Any disease NB01 flagged near-chance had a within-host accuracy barely above guessing. Its gap is still computed, but the gap of a disease the classifier couldn't do *within* host doesn't measure transfer — there was no working classifier to transfer. The flag travels with the disease into the table so next week's interpretation reads that gap with the caveat attached.

In [ ]:
import pandas as pd

# Pair each transfer subject's within-host accuracy (loaded from NB01 in
# Section 1) with its cross-host accuracy (Section 3) and subtract. The
# near-chance flag from NB01 travels with each disease.
rows = []
for d in transfer_diseases:
    wh = within_host_acc[d]            # from NB01's within_host_accuracy.parquet
    ch = cross_host_accuracy[d]        # from Section 3
    rows.append({
        "disease":         d,
        "within_host_acc": float(wh),
        "cross_host_acc":  float(ch),
        "gap":             float(wh - ch),
        "n_cross_host":    int(cross_host_n[d]),
        "within_host_flag": within_host_flag[d],   # "ok" / "near-chance"
    })

# Per-disease rows are built ALWAYS. The aggregation precommit decides only
# whether to ALSO append an overall row — so the per-disease core is identical
# whatever you chose, and NB03 always finds it.
if agg_level in ("overall", "both"):
    overall_wh = within_host_summary["results"]["overall_within_host_acc"]
    rows.append({
        "disease":          "overall",
        "within_host_acc":  float(overall_wh),
        "cross_host_acc":   float(overall_cross_host_acc),
        "gap":              float(overall_wh - overall_cross_host_acc),
        "n_cross_host":     int(len(y_true)),
        "within_host_flag": "n/a",      # the flag is a per-disease judgment
    })

gap_table = pd.DataFrame(
    rows,
    columns=["disease", "within_host_acc", "cross_host_acc", "gap",
             "n_cross_host", "within_host_flag"],
)

print(f"Gap table: {len(transfer_diseases)} per-disease row(s)"
      + (" + 1 overall row" if agg_level in ("overall", "both") else "")
      + f"   (aggregation: {agg_level})")

In [ ]:
# Read the gap table back. The per-disease gaps are the result; an overall row,
# if present, is context. A near-chance flag means: do not read this disease's
# gap as a transfer measurement — there was no working within-host classifier to
# transfer from.
print(f"{'disease':18s} {'within':>8s} {'cross':>8s} {'gap':>8s} "
      f"{'n':>7s}  {'flag':<12s}")
for _, r in gap_table.iterrows():
    caveat = "  <- read with caution" if r["within_host_flag"] == "near-chance" else ""
    print(f"{r['disease']:18s} {r['within_host_acc']:>8.3f} "
          f"{r['cross_host_acc']:>8.3f} {r['gap']:>8.3f} {r['n_cross_host']:>7,}  "
          f"{r['within_host_flag']:<12s}{caveat}")

flagged_here = [r["disease"] for _, r in gap_table.iterrows()
                if r["within_host_flag"] == "near-chance"]
if flagged_here:
    print(f"\n{len(flagged_here)} disease(s) carry a near-chance caveat: "
          f"{', '.join(flagged_here)}.")
    print("Their gaps are in the table but don't measure transfer — next week's "
          "interpretation handles them as such.")
else:
    print("\nNo near-chance caveats — every gap rests on a working within-host "
          "classifier.")

if DEV_MODE:
    print("\nDEV_MODE is on — this whole table is a dev-run rehearsal. The real "
          "gaps come from a DEV_MODE = False run matching NB01.")

### 5) Inspect the per-disease gap distribution (which diseases transfer; which don't)

You have a gap for every transfer subject. Before saving, look at them together — the shape of the per-disease gaps is the finding this notebook hands to next week.

The question to ask of the picture: do the diseases land together, or do they spread? If every gap is small, transfer held up across the board — the classifier mostly learned diseases, not hosts. If the gaps spread — some near zero, some large — then transfer is disease-specific, and *which* diseases fall where is the thing next week's interpretation is built on. The page predicts the spread, not the uniform case.

What you're doing here is ranking and describing, not explaining. *Why* a particular disease has a large gap — whether the classifier leaned on the host's background, or the disease genuinely looks different on the withheld host — is next week's work, and it needs information this notebook doesn't have. For now: which diseases transferred, which didn't, and how far apart they are.

In [ ]:
import matplotlib.pyplot as plt

# The per-disease gaps only (exclude any overall row), sorted largest-first so
# the diseases that transferred worst sit at the top. A small multiple, because
# there are only a few transfer subjects — this is the whole distribution, not a
# sample of it.
per_disease = gap_table[gap_table["disease"] != "overall"].copy()
per_disease = per_disease.sort_values("gap", ascending=True)  # ascending -> largest at top in barh

fig, ax = plt.subplots(figsize=(7, 0.6 * len(per_disease) + 1.2))
colors = ["tab:red" if f == "near-chance" else "tab:blue"
          for f in per_disease["within_host_flag"]]
ax.barh(per_disease["disease"], per_disease["gap"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("within-host − cross-host accuracy  (gap)")
ax.set_title("Per-disease transfer gap (larger = transferred worse)")

# Mark the near-chance diseases so a large gap that doesn't mean transfer is
# visually distinct from one that does.
for y, (_, r) in enumerate(per_disease.iterrows()):
    if r["within_host_flag"] == "near-chance":
        ax.text(r["gap"], y, "  near-chance (within host)",
                va="center", fontsize=8, color="tab:red")

plt.tight_layout()
plt.show()

# The same ordering as a table, smallest gap (best transfer) first.
print(f"{'disease':18s} {'gap':>8s}  {'flag':<12s}")
for _, r in per_disease.sort_values("gap").iterrows():
    print(f"{r['disease']:18s} {r['gap']:>8.3f}  {r['within_host_flag']:<12s}")

This picture is the input to next week. NB03 takes each disease's gap and assigns it a reading — but it does so along *two* axes, not one. The gap (did transfer hold up?) is the axis you measured here. The second axis is pathogen identity: is the disease caused by the same pathogen on both hosts, or by different pathogens that happen to share a name? The two axes together separate four cases — a clean transfer, a classifier that learned generic leaf appearance, a host-template shortcut, and two genuinely different diseases under one label — that the gap alone can't tell apart. That's why the interpretation is its own notebook.

What you're carrying into it is exactly what's in the gap table: for each disease, its within-host accuracy, its cross-host accuracy, the gap between them, and whether the within-host number was sound enough to trust the gap at all. The closeout saves that, plus a short summary, and that's Week 3 done.

### 6) Closeout: save `cross_host_accuracy.parquet`, `gap_table.parquet`, and `comparison_summary.json`; submit draft presentation

Week 3 is done once this notebook has saved what next week reads. Three files:

- `cross_host_accuracy.parquet` — per-disease cross-host accuracy on its own, the companion to the within-host table from Week 2.
- `gap_table.parquet` — the per-disease within-host vs cross-host gap, with the near-chance flag carried alongside. This is the file next week's interpretation is built on.
- `comparison_summary.json` — a short record in the same shape as Week 2's summary: what you found (the gaps), how you measured it (the design), and a one-line reading. It carries the label order so next week can confirm it's working with the same diseases.

Each save reloads the file and checks it matches what's in memory, the same round-trip habit from Week 2 — a file that didn't write correctly should fail here, not next week.

In [ ]:
# 1. The standalone per-disease cross-host accuracy table.
cross_host_df = pd.DataFrame(
    [{"disease": d,
      "cross_host_acc": float(cross_host_accuracy[d]),
      "n_cross_host": int(cross_host_n[d])}
     for d in transfer_diseases],
    columns=["disease", "cross_host_acc", "n_cross_host"],
)
cross_host_path = OUTPUT_DIR / "cross_host_accuracy.parquet"
cross_host_df.to_parquet(cross_host_path, index=False)
reloaded = pd.read_parquet(cross_host_path)
assert reloaded.equals(cross_host_df), "cross_host_accuracy.parquet didn't round-trip."
print(f"Wrote {cross_host_path}  ({len(cross_host_df)} rows)")

# 2. The gap table (built in Section 4).
gap_path = OUTPUT_DIR / "gap_table.parquet"
gap_table.to_parquet(gap_path, index=False)
reloaded = pd.read_parquet(gap_path)
assert reloaded.equals(gap_table), "gap_table.parquet didn't round-trip."
print(f"Wrote {gap_path}  ({len(gap_table)} rows)")

In [ ]:
import json

# A compact record in the R2 results / methodology / interpretation shape, the
# same shape Week 2 used. The interpretation line is a plain reading of the
# spread — not the per-disease verdicts, which are next week's job.
per_disease_rows = gap_table[gap_table["disease"] != "overall"]
overall_gap = float(within_host_summary["results"]["overall_within_host_acc"]
                    - overall_cross_host_acc)

comparison_summary = {
    "results": {
        "overall_within_host_acc": float(
            within_host_summary["results"]["overall_within_host_acc"]),
        "overall_cross_host_acc":  float(overall_cross_host_acc),
        "overall_gap":             overall_gap,
        "per_disease_gap": {
            r["disease"]: float(r["gap"]) for _, r in per_disease_rows.iterrows()
        },
        "per_disease_cross_host_acc": {
            d: float(cross_host_accuracy[d]) for d in transfer_diseases
        },
        "flagged_diseases": [
            r["disease"] for _, r in per_disease_rows.iterrows()
            if r["within_host_flag"] == "near-chance"
        ],
    },
    "methodology": {
        "label_space": "disease",
        "diseases": diseases,                    # label order — NB03 asserts on this
        "transfer_diseases": transfer_diseases,  # the subset with a withheld host
        "num_classes": num_classes,
        "hold_out_scheme": scheme,
        "held_out_host": {d: held_out[d] for d in transfer_diseases},
        "aggregation": agg_level,
        "dev_mode": DEV_MODE,
    },
    "interpretation": (
        f"Measured cross-host transfer for {len(transfer_diseases)} disease(s) "
        f"under the {scheme} scheme. Overall gap (within − cross): "
        f"{overall_gap:.3f}. Per-disease gaps are in gap_table.parquet; their "
        "spread, and the reading of each, is the Week-4 interpretation."
    ),
}

summary_path = OUTPUT_DIR / "comparison_summary.json"
with open(summary_path, "w") as f:
    json.dump(comparison_summary, f, indent=2)

with open(summary_path) as f:
    reloaded = json.load(f)
assert reloaded == comparison_summary, "comparison_summary.json didn't round-trip."
assert reloaded["methodology"]["diseases"] == diseases, "label order drift on save."
print(f"Wrote {summary_path}")

Your Week-3 deliverable is these three files plus a **draft presentation**. The presentation reports what you measured: the per-disease transfer gaps, which diseases held up across hosts and which didn't, and the overall picture. Frame it as a measurement, not a verdict — you can say a disease's accuracy dropped on the withheld host, but *why* it dropped is next week's interpretation, and saying it now would get ahead of the evidence.

Two things to keep honest in the draft: report the gaps per disease, not as one averaged number (an average hides the spread that is the finding), and flag any near-chance disease explicitly, since its gap doesn't measure transfer. Next week (NB03) takes this gap table, places each disease in the transfer-outcome × pathogen-identity grid, and writes the per-disease interpretation that becomes the paper. Submit your three files and your draft presentation.